# Hyper-Adaptive Momentum Dynamics for Native Cubic Portfolio Optimization

## Strategy Description
This notebook implements the Hyper-Adaptive Momentum Dynamics (HAMD) strategy for cubic cardinality-constrained portfolio optimization as described in the paper by Greg Serbarinov. The strategy aims to optimize a portfolio by directly operating on the native higher-order objective using a hybrid pipeline combining continuous Hamiltonian search, exact cardinality-preserving projection, and iterated local search (ILS).

**Paper Citation:**
Serbarinov, G. (2026). Hyper-Adaptive Momentum Dynamics for Native Cubic Portfolio Optimization: Avoiding Quadratization Distortion in Higher-Order Cardinality-Constrained Search. arXiv preprint arXiv:2603.15947.

**Abstract:**
The paper studies cubic cardinality-constrained portfolio optimization, where three-way sector co-movement terms augment the quadratic risk-return objective. HAMD operates directly on the native higher-order objective, achieving substantial improvements over traditional methods like simulated annealing and tabu search.


In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## Phase 1 — Trading Context & Objectives

In [ ]:
# Configuration
UNIVERSE = ['AAPL', 'MSFT']
RISK_FREE_RATE = 0.02
LOOKBACK_PERIOD = 252
REBALANCING_PERIOD = 'M'
POSITION_SIZE = 0.02

# Hypothesis
# The strategy hypothesizes that by directly optimizing the native cubic objective,
# we can achieve better portfolio performance compared to traditional quadratized approaches.


## Phase 2 — Data Download & Feature Computation

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

# Download data
data = yf.download(UNIVERSE, start='2010-01-01', end='2023-01-01', interval='1d')
prices = data['Adj Close']

# Compute returns
returns = prices.pct_change().dropna()

# Cross-sectional normalization
normalized_returns = (returns - returns.mean(axis=1, skipna=True).values.reshape(-1, 1)) / returns.std(axis=1, skipna=True).values.reshape(-1, 1)

## Phase 3 — Signal Generation & Portfolio Construction

In [ ]:
# Signal generation
momentum_signals = normalized_returns.rolling(window=LOOKBACK_PERIOD).mean()

# Position sizing
positions = momentum_signals.rank(axis=1, ascending=False, pct=True)
positions = positions.sub(positions.mean(axis=1), axis=0)
positions = positions.div(positions.abs().sum(axis=1), axis=0)

# Portfolio construction
portfolio_weights = positions.mul(POSITION_SIZE)

## Phase 4 — Vectorized Backtest

In [ ]:
# Shift signals forward by 1 period to avoid look-ahead bias
portfolio_weights_shifted = portfolio_weights.shift(1)

# Compute portfolio returns
portfolio_returns = (returns * portfolio_weights_shifted).sum(axis=1)

## Phase 5 — Performance Metrics

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import norm

# Compute cumulative returns
cumulative_returns = (1 + portfolio_returns).cumprod()

# Performance metrics
annual_return = portfolio_returns.mean() * 252
annual_volatility = portfolio_returns.std() * np.sqrt(252)
sharpe_ratio = (annual_return - RISK_FREE_RATE) / annual_volatility
sortino_ratio = (annual_return - RISK_FREE_RATE) / portfolio_returns[portfolio_returns < 0].std() * np.sqrt(252)
max_drawdown = (cumulative_returns.cummax() - cumulative_returns).max()
calmar_ratio = annual_return / max_drawdown

# Print metrics
print(f'Annual Return: {annual_return:.2%}')
print(f'Annual Volatility: {annual_volatility:.2%}')
print(f'Sharpe Ratio: {sharpe_ratio:.2f}')
print(f'Sortino Ratio: {sortino_ratio:.2f}')
print(f'Max Drawdown: {max_drawdown:.2%}')
print(f'Calmar Ratio: {calmar_ratio:.2f}')

# Plot equity curve
plt.plot(cumulative_returns)
plt.title('Equity Curve')
plt.xlabel('Date')
plt.ylabel('Cumulative Returns')
plt.show()

## Phase 6 — Monitoring Stub

In [ ]:
def monitor_portfolio(live_data):
    # Placeholder function for monitoring
    live_returns = live_data.pct_change().dropna()
    live_positions = momentum_signals.iloc[-1]
    daily_pnl = (live_returns * live_positions).sum()
    print(f'Daily P&L: {daily_pnl:.2%}')
    print(f'Current Positions: {live_positions.to_dict()}')

# Example usage
monitor_portfolio(prices.iloc[-1])